# INSTALL

In [1]:
!pip install typhoon-ocr pdf2image Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 19.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [typhoon-ocr] [openai]c]


In [2]:
!apt-get install -y poppler-utils

zsh:1: command not found: apt-get


# IMPORT

In [5]:
import os
import json
import re
from typhoon_ocr import ocr_document

# Pipeline

In [ ]:
class Extrator:
    #  Initialize
    def __init__(self, api_key: str):
        """Initializes the OCR engine with the provided API key."""
        os.environ["TYPHOON_OCR_API_KEY"] = api_key
    
    def

In [ ]:
from typhoon_ocr import ocr_document

# OCR each page
raw_texts = []
for i in range(len(images)):
    print(f"Processing page {i+1}...")
    markdown = ocr_document(f"page_{i+1}.jpg")
    raw_texts.append(markdown)
    print(markdown)  # preview

Processing page 1...
ส.ส. ๕/๑๘ (บช)
สีขาว

# รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ

ตามที่ได้มีพระราชกฤษฎีกาให้มีภารเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง ได้กำหนดให้วันที่ 8 เดือน กุมภาพันธ์ พ.ศ. 2569 เป็นวันเลือกตั้ง นั้น

บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อของหน่วยเลือกตั้งที่ 1 หมู่ที่ 1 ตำบล/แขวง/เทศบาล ไม่น้อย อำเภอ/เขต เมือง เขตเลือกตั้งที่ 2 จังหวัด อุบลราชธานี เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของหน่วยเลือกตั้งดังกล่าว ดังนี้

๑. จำนวนผู้มีสิทธิเลือกตั้ง
๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 705 คน (เจ็ดร้อยเจ็ด)

๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 513 คน (ห้าร้อยสิบสาม)

๒. จำนวนบัตรเลือกตั้ง
๒.๑ จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร จำนวน 700 บัตร (เจ็ดร้อย)

๒.๒ จำนวนบัตรเลือกตั้งที่ใช้ จำนวน 513 บัตร (ห้าร้อยสิบสาม)

๒.๒.๑ บัตรดี จำนวน 468 บัตร (สี่ร้อยหกสิบแปด)

๒.๒.๒ บัตรเสีย จำนวน 20 บัตร (ยี่สิบ)

In [ ]:
import json
import re

def parse_typhoon_markdown(raw_pages: list[str], source_file: str) -> dict:
    full_text = "\n".join(raw_pages)

    # --- Extract header info ---
    province = re.search(r'จังหวัด\s*(\S+)', full_text)
    constituency = re.search(r'เขตเลือกตั้งที่\s*(\d+)', full_text)
    district = re.search(r'อำเภอ/เขต\s*(\S+)', full_text)
    subdistrict = re.search(r'ตำบล/แขวง/เทศบาล\s*(\S+)', full_text)
    unit = re.search(r'หน่วยเลือกตั้งที่\s*(\d+)', full_text)

    # --- Extract summary numbers ---
    def extract_num(pattern):
        m = re.search(pattern, full_text)
        return int(m.group(1).replace(',', '')) if m else None

    summary = {
        "1.1": extract_num(r'๑\.๑.*?จำนวน\s+(\d+)\s+คน'),
        "1.2": extract_num(r'๑\.๒.*?จำนวน\s+(\d+)\s+คน'),
        "2.1": extract_num(r'๒\.๑.*?จำนวน\s+(\d+)\s+บัตร'),
        "2.2": extract_num(r'๒\.๒\s+จำนวนบัตรเลือกตั้งที่ใช้.*?จำนวน\s+(\d+)\s+บัตร'),
        "2.2.1": extract_num(r'๒\.๒\.๑.*?จำนวน\s+(\d+)\s+บัตร'),
        "2.2.2": extract_num(r'๒\.๒\.๒.*?จำนวน\s+(\d+)\s+บัตร'),
        "2.2.3": extract_num(r'๒\.๒\.๓.*?จำนวน\s+(\d+)\s+บัตร'),
        "2.3": extract_num(r'๒\.๓.*?จำนวน\s+(\d+)\s+บัตร'),
    }

    # --- Extract party results from HTML tables ---
    results = []
    # Match table rows: <tr><td>NUMBER</td><td>PARTY</td><td>VOTES (...)</td></tr>
    row_pattern = re.compile(
        r'<tr><td>([๐-๙\d]+)</td><td>(.+?)</td><td>(\d+)\s*\([^)]+\)</td></tr>'
    )

    def thai_to_arabic(thai_num: str) -> int:
        thai_digits = '๐๑๒๓๔๕๖๗๘๙'
        return int(''.join(str(thai_digits.index(c)) if c in thai_digits else c for c in thai_num))

    for match in row_pattern.finditer(full_text):
        num_str, party, votes_str = match.groups()
        # Skip total row
        if 'รวม' in num_str:
            continue
        results.append({
            "number": thai_to_arabic(num_str),
            "party": party.strip(),
            "votes": int(votes_str)
        })

    # Sort by party number
    results.sort(key=lambda x: x["number"])

    # --- Validation ---
    vote_sum = sum(r["votes"] for r in results)
    good_ballots = summary.get("2.2.1", 0)
    diff_pct = abs(vote_sum - good_ballots) / good_ballots * 100 if good_ballots else 0

    return {
        "province_name": province.group(1) if province else None,
        "constituency_number": int(constituency.group(1)) if constituency else None,
        "district": district.group(1) if district else None,
        "subdistrict": subdistrict.group(1) if subdistrict else None,
        "polling_unit": int(unit.group(1)) if unit else None,
        "form_type": "party_list",
        "election_date": "2026-02-08",
        "summary": summary,
        "results": results,
        "_source_file": source_file,
        "_validation": {
            "vote_sum": vote_sum,
            "good_ballots": good_ballots,
            "sum_diff_pct": round(diff_pct, 2),
            "status": "✅ PASS" if diff_pct < 1 else "⚠️ MISMATCH"
        }
    }

In [ ]:
result = parse_typhoon_markdown(raw_texts, source_file=pdf_filename)

print(json.dumps(result, ensure_ascii=False, indent=2))

with open("output.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

files.download("output.json")

{
  "province_name": "อุบลราชธานี",
  "constituency_number": 2,
  "district": "เมือง",
  "subdistrict": "ไม่น้อย",
  "polling_unit": 1,
  "form_type": "party_list",
  "election_date": "2026-02-08",
  "summary": {
    "1.1": 705,
    "1.2": 513,
    "2.1": 700,
    "2.2": 513,
    "2.2.1": 468,
    "2.2.2": 20,
    "2.2.3": 25,
    "2.3": 25
  },
  "results": [
    {
      "number": 1,
      "party": "ไทยทรัพย์ทวี",
      "votes": 6
    },
    {
      "number": 2,
      "party": "เพื่อชาติไทย",
      "votes": 7
    },
    {
      "number": 3,
      "party": "ใหม่",
      "votes": 4
    },
    {
      "number": 4,
      "party": "มิติใหม่",
      "votes": 2
    },
    {
      "number": 5,
      "party": "รวมใจไทย",
      "votes": 9
    },
    {
      "number": 6,
      "party": "รวมไทยสร้างชาติ",
      "votes": 3
    },
    {
      "number": 7,
      "party": "พลวัต",
      "votes": 3
    },
    {
      "number": 8,
      "party": "ประชาธิปไตยใหม่",
      "votes": 3
    },
    {
      "n

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>